In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [4]:
import sys
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import BertTokenizer

sys.path.insert(0, r"d:\study\ml-foundations\bert")
from paralla_decoder import BertEncoder, ParallelDecoder

using: cuda


In [5]:
class SequencePooler(nn.Module):
    """Mean-pools BERT hidden states [B, seq_len, hidden_dim] → [B, latent_dim]."""
    def __init__(self, hidden_dim=768, latent_dim=256):
        super().__init__()
        self.proj = nn.Linear(hidden_dim, latent_dim)

    def forward(self, hidden_states, attention_mask=None):
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden_states * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        else:
            pooled = hidden_states.mean(1)
        return self.proj(pooled)   # [B, latent_dim]

In [6]:
encoder = BertEncoder().to(device)
decoder = ParallelDecoder(latent_dim=256).to(device)
print("encoder and decoder ready")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


encoder and decoder ready


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MetricNet(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.SiLU(),
            nn.Linear(512, 512),
            nn.SiLU(),
            nn.Linear(512, latent_dim * latent_dim)
        )
    
    def forward(self, z):
        B = z.shape[0]
        L_flat = self.net(z)                                    # [B, d*d]
        L = L_flat.view(B, self.latent_dim, self.latent_dim)   # [B, d, d]
        
        # enforce lower triangular
        L = torch.tril(L)
        
        # enforce positive diagonal via softplus
        diag_idx = torch.arange(self.latent_dim)
        L = L.clone()
        L[:, diag_idx, diag_idx] = F.softplus(L[:, diag_idx, diag_idx]) + 1e-4
        
        # SPD matrix
        G = L @ L.transpose(-2, -1)                            # [B, d, d]
        return G, L


# ── test ──────────────────────────────────────────────────────────────
metric_net = MetricNet(latent_dim=256).to(device)

z_test = torch.randn(2, 256).to(device)
G, L = metric_net(z_test)

eigenvalues = torch.linalg.eigvalsh(G)

print(f"G shape:        {G.shape}")           # [2, 256, 256]
print(f"L shape:        {L.shape}")           # [2, 256, 256]
print(f"min eigenvalue: {eigenvalues.min().item():.6f}")  # must be > 0
print(f"max eigenvalue: {eigenvalues.max().item():.6f}")
print(f"SPD check:      {'PASSED' if eigenvalues.min() > 0 else 'FAILED'}")

G shape:        torch.Size([2, 256, 256])
L shape:        torch.Size([2, 256, 256])
min eigenvalue: 0.010706
max eigenvalue: 3.544635
SPD check:      PASSED


In [11]:
class FlowNet(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, 512),  # +1 for time t
            nn.SiLU(),
            nn.Linear(512, 512),
            nn.SiLU(),
            nn.Linear(512, 512),
            nn.SiLU(),
            nn.Linear(512, latent_dim)
        )
    
    def forward(self, z_t, t):
        # z_t: [B, latent_dim]
        # t:   [B]
        t_expand = t.unsqueeze(-1)                  # [B, 1]
        inp      = torch.cat([z_t, t_expand], dim=-1)  # [B, latent_dim+1]
        return self.net(inp)                        # [B, latent_dim]


# ── test ──────────────────────────────────────────────────────────────
flow_net = FlowNet(latent_dim=256).to(device)

z_t_test = torch.randn(2, 256).to(device)
t_test   = torch.rand(2).to(device)

v_pred = flow_net(z_t_test, t_test)

print(f"z_t shape: {z_t_test.shape}")  # [2, 256]
print(f"t shape:   {t_test.shape}")    # [2]
print(f"v_pred shape: {v_pred.shape}") # [2, 256]
print("FlowNet check: PASSED" if v_pred.shape == z_t_test.shape else "FAILED")

z_t shape: torch.Size([2, 256])
t shape:   torch.Size([2])
v_pred shape: torch.Size([2, 256])
FlowNet check: PASSED


In [ ]:
checkpoint = torch.load("stage1_best.pt", map_location=device)
decoder.load_state_dict(checkpoint["decoder"])
for param in decoder.parameters():
    param.requires_grad = False
print("stage1 decoder loaded and frozen")

In [ ]:
from paralla_decoder import build_dataloaders

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
train_loader, val_loader = build_dataloaders(tokenizer, train_size=200000)

In [ ]:
# stage 2: sequence-level flow matching
# z_data = decoder.compress(encoder_hidden) → [B, 128, 256] per-position latents
# flow net learns noise [B*128, 256] → data [B*128, 256], position-wise

flow_net  = FlowNet(latent_dim=256).to(device)
optimizer = AdamW(flow_net.parameters(), lr=1e-4)

EPOCHS    = 20
best_loss = float("inf")

for epoch in range(EPOCHS):
    flow_net.train()
    encoder.eval()

    train_loss = 0

    for step, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with torch.no_grad():
            hidden = encoder(input_ids, attention_mask)  # [B, 128, 768]
            z_data = decoder.compress(hidden)             # [B, 128, 256]

        B, S, D = z_data.shape
        z_flat  = z_data.view(B * S, D)                  # [B*128, 256]
        z_noise = torch.randn_like(z_flat)

        t   = torch.rand(B * S, device=device)
        z_t = (1 - t.unsqueeze(-1)) * z_noise + t.unsqueeze(-1) * z_flat

        v_true = z_flat - z_noise                        # [B*128, 256]
        v_pred = flow_net(z_t, t)                        # [B*128, 256]

        loss = F.mse_loss(v_pred, v_true)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(flow_net.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()

        if step % 50 == 0:
            print(f"epoch {epoch+1} step {step}/{len(train_loader)} | loss {loss.item():.4f}")

    avg_loss = train_loss / len(train_loader)
    print(f"\nepoch {epoch+1} done | avg loss {avg_loss:.4f}\n")

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({"flow_net": flow_net.state_dict()}, "stage2_best.pt")
        print(f"saved best model at loss {best_loss:.4f}")

In [ ]:
@torch.no_grad()
def generate(flow_net, decoder, tokenizer, steps=200, num_samples=5):
    flow_net.eval()
    decoder.eval()

    for i in range(num_samples):
        z_flat = torch.randn(128, 256, device=device)    # [128, 256]

        dt = 1.0 / steps
        for step in range(steps):
            t = torch.full((128,), step * dt, device=device)
            v = flow_net(z_flat, t)
            z_flat = z_flat + v * dt

        z_seq = z_flat.unsqueeze(0)                      # [1, 128, 256]

        x        = decoder.project_up(z_seq)             # [1, 128, 768]
        out      = decoder.bert(inputs_embeds=x)
        logits   = decoder.to_logits(out.last_hidden_state)
        pred_ids = logits.argmax(-1)

        text = tokenizer.decode(pred_ids[0].cpu(), skip_special_tokens=True)
        print(f"sample {i+1}: {text[:200]}")
        print()

generate(flow_net, decoder, tokenizer)